In [1]:
!pip install -q transformers datasets accelerate evaluate jiwer tensorboard
!pip install -q soundfile librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 41.6 MB/s eta 0:00:0000:0100:01


In [2]:
# ── Cell 1: Imports ──────────────────────────────────────────────────────────
import os
import gc
import torch
import pandas as pd
import numpy as np
import evaluate
import soundfile as sf
import librosa
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import Dataset, Audio
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🔥 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🔥 GPU: {torch.cuda.get_device_name(0)}")
    print(f"🔥 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

🔥 PyTorch version: 2.9.0+cu126
🔥 CUDA available: True
🔥 GPU: Tesla T4
🔥 GPU Memory: 15.6 GB


In [3]:
class Config:
    # Base dataset path
    DATASET_BASE = "/kaggle/input/datasets/panditaadarsh/nepali-english-codeswitched"

    # Correct audio folder
    AUDIO_DIR = os.path.join(
        DATASET_BASE,
        "kaggle_upload",
        "audios_segment"
    )

    # Correct metadata CSV
    METADATA_CSV = "/kaggle/input/datasets/panditaadarsh/codeswitchv3/metadata_cycle1.csv"

    # Output
    OUTPUT_DIR = "/kaggle/working/whisper-small-nepali-english-cs"

    # Model
    MODEL_NAME = "/kaggle/input/notebooks/panditaadarsh/finetuning-the-codeswitched-whisper/whisper-small-nepali-english-cs/final-model"
    LANGUAGE = None
    TASK = "transcribe"

    # Training
    BATCH_SIZE = 2
    GRADIENT_ACCUMULATION = 8
    LEARNING_RATE = 1e-5
    WARMUP_STEPS = 300
    NUM_EPOCHS = 5
    EVAL_STEPS = 500
    SAVE_STEPS = 500
    LOGGING_STEPS = 50
    FP16 = True

    # Audio
    SAMPLING_RATE = 16000
    MAX_AUDIO_LENGTH_SEC = 30

    # Split
    TEST_SIZE = 0.05
    SEED = 42

config = Config()

print("✅ Config loaded")
print("CSV:", config.METADATA_CSV)
print("Audio dir:", config.AUDIO_DIR)

✅ Config loaded
CSV: /kaggle/input/datasets/panditaadarsh/codeswitchv3/metadata_cycle1.csv
Audio dir: /kaggle/input/datasets/panditaadarsh/nepali-english-codeswitched/kaggle_upload/audios_segment


In [4]:
# ── Cell 3: Load & Prepare Metadata (Robust Version) ───────────────────────

import os
import pandas as pd
import csv

print("📂 Loading metadata...")

# --- Safe CSV loading (handles broken rows gracefully) ---
df = pd.read_csv(
    config.METADATA_CSV,
    engine="python",          # more tolerant than default C engine
    on_bad_lines="warn",      # change to "skip" if you want silent cleanup
    quoting=csv.QUOTE_MINIMAL # helps with messy text fields
)

print(f"   Total samples (raw): {len(df)}")
print(f"   Columns: {list(df.columns)}")
print(f"   Sample:\n{df.head(3)}")

# --- Validate required columns ---
assert "path" in df.columns, "CSV must have 'path' column"
assert "transcription" in df.columns, "CSV must have 'transcription' column"

# --- Resolve audio paths ---
def resolve_audio_path(relative_path):
    filename = os.path.basename(str(relative_path))
    return os.path.join(config.AUDIO_DIR, filename)

df["audio_path"] = df["path"].apply(resolve_audio_path)

# --- Remove missing audio files ---
missing_mask = ~df["audio_path"].apply(os.path.exists)
missing_count = missing_mask.sum()

if missing_count > 0:
    print(f"⚠️  {missing_count} audio files not found! Removing them.")
    df = df[~missing_mask].reset_index(drop=True)

print(f"✅ Valid samples after file check: {len(df)}")

# --- Clean transcription column safely ---
df["transcription"] = df["transcription"].astype(str)

# Remove NaN-like strings and empty text
df = df.dropna(subset=["transcription"])
df = df[df["transcription"].str.strip() != ""].reset_index(drop=True)

# Optional: normalize whitespace
df["transcription"] = df["transcription"].str.replace(r"\s+", " ", regex=True).str.strip()

print(f"✅ Final cleaned samples: {len(df)}")
print("🎯 Data ready for training!")

📂 Loading metadata...
   Total samples (raw): 9995
   Columns: ['path', 'transcription']
   Sample:
                             path  \
0  audios_segment/audio_00001.wav   
1  audios_segment/audio_00002.wav   
2  audios_segment/audio_00003.wav   

                                       transcription  
0  खेल्न थाल्यो। म academy wise खेले अ त्यसरी प्र...  
1  वर्ष भयो होला मैले त्यो filmमा हेरेको तपाईले य...  
2  अब मेरो एउटा dream चाहिँ कस्तो छ भनेछि अब fina...  


/tmp/ipykernel_57/4008071039.py:10: ParserWarning: Skipping line 6397: Expected 2 fields in line 6397, saw 4

  df = pd.read_csv(
/tmp/ipykernel_57/4008071039.py:10: ParserWarning: Skipping line 6583: Expected 2 fields in line 6583, saw 4

  df = pd.read_csv(
/tmp/ipykernel_57/4008071039.py:10: ParserWarning: Skipping line 8331: Expected 2 fields in line 8331, saw 3

  df = pd.read_csv(
/tmp/ipykernel_57/4008071039.py:10: ParserWarning: Skipping line 9177: Expected 2 fields in line 9177, saw 4

  df = pd.read_csv(
/tmp/ipykernel_57/4008071039.py:10: ParserWarning: Skipping line 9670: Expected 2 fields in line 9670, saw 3

  df = pd.read_csv(


✅ Valid samples after file check: 9995
✅ Final cleaned samples: 9995
🎯 Data ready for training!


In [5]:
# ── Cell 4: Create HuggingFace Dataset ──────────────────────────────────────
print("📦 Creating HuggingFace Dataset...")

dataset = Dataset.from_dict({
    "audio": df["audio_path"].tolist(),
    "transcription": df["transcription"].tolist(),
}).cast_column("audio", Audio(sampling_rate=config.SAMPLING_RATE))

dataset = dataset.train_test_split(
    test_size=config.TEST_SIZE,
    seed=config.SEED
)
print(f"✅ Train: {len(dataset['train'])} | Validation: {len(dataset['test'])}")

📦 Creating HuggingFace Dataset...
✅ Train: 9495 | Validation: 500


In [6]:
print(f"🤖 Loading {config.MODEL_NAME}...")

feature_extractor = WhisperFeatureExtractor.from_pretrained(config.MODEL_NAME)

tokenizer = WhisperTokenizer.from_pretrained(
    config.MODEL_NAME, language="ne", task="transcribe",
)
processor = WhisperProcessor.from_pretrained(
    config.MODEL_NAME, language="ne", task="transcribe",
)

model = WhisperForConditionalGeneration.from_pretrained(config.MODEL_NAME)

model.generation_config.language = "ne"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="ne", task="transcribe",
)
model.generation_config.suppress_tokens = []           # English allowed
model.generation_config.max_new_tokens = 225           # Stop hallucination
model.generation_config.no_repeat_ngram_size = 3       # Kill repeat loops

print(f"✅ Model parameters: {model.num_parameters() / 1e6:.1f}M")
print("✅ Base: Nepali | English: Allowed | Anti-hallucination: ON")

🤖 Loading /kaggle/input/notebooks/panditaadarsh/finetuning-the-codeswitched-whisper/whisper-small-nepali-english-cs/final-model...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

✅ Model parameters: 241.7M
✅ Base: Nepali | English: Allowed | Anti-hallucination: ON


In [7]:
# ── Cell 6: Preprocessing ───────────────────────────────────────────────────
def prepare_dataset(batch):
    audio = batch["audio"]

    batch["input_features"] = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = tokenizer(batch["transcription"]).input_ids

    return batch

print("⚙️  Preprocessing datasets (this may take a while)...")
dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset.column_names["train"],
    num_proc=1,
)
print("✅ Preprocessing complete!")

⚙️  Preprocessing datasets (this may take a while)...


Map:   0%|          | 0/9495 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

✅ Preprocessing complete!


In [8]:
# ── Cell 7: Data Collator ───────────────────────────────────────────────────
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)
print("✅ Data collator ready")

✅ Data collator ready


In [9]:
# ── Cell 8: Evaluation Metric (WER) ─────────────────────────────────────────
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

print("✅ WER metric loaded")

✅ WER metric loaded


In [ ]:
class WERLoggingTrainer(Seq2SeqTrainer):
    """Seq2SeqTrainer that also logs sampled WER alongside training loss."""

    def __init__(self, *args, wer_samples=None, wer_tokenizer=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.wer_samples = wer_samples      # list of preprocessed dicts
        self.wer_tokenizer = wer_tokenizer

    def log(self, logs):
        # If this is a training-loss log and we have WER samples, compute WER
        if self.wer_samples and "loss" in logs and self.model is not None:
            self.model.eval()
            preds_text, refs_text = [], []
            for s in self.wer_samples:
                inp = torch.tensor(s["input_features"]).unsqueeze(0).to(self.model.device)
                with torch.no_grad():
                    ids = self.model.generate(inp, max_new_tokens=225)
                preds_text.append(self.wer_tokenizer.decode(ids[0], skip_special_tokens=True))
                lab = [t if t != -100 else self.wer_tokenizer.pad_token_id for t in s["labels"]]
                refs_text.append(self.wer_tokenizer.decode(lab, skip_special_tokens=True))
            logs["sampled_wer"] = round(
                100 * wer_metric.compute(predictions=preds_text, references=refs_text), 2
            )
            self.model.train()
        super().log(logs)

# Pre-load a small fixed sample from the eval set for fast WER estimation
rng = np.random.RandomState(config.SEED)
_idx = rng.choice(len(dataset["test"]), config.WER_SAMPLE_SIZE, replace=False)
wer_samples = [dataset["test"][int(i)] for i in _idx]

print(f"✅ WERLoggingTrainer ready  ({config.WER_SAMPLE_SIZE} samples for live WER)")


In [10]:
training_args = Seq2SeqTrainingArguments(
    output_dir=config.OUTPUT_DIR,
    per_device_train_batch_size=config.BATCH_SIZE,
    per_device_eval_batch_size=config.BATCH_SIZE,
    gradient_accumulation_steps=config.GRADIENT_ACCUMULATION,
    learning_rate=config.LEARNING_RATE,
    warmup_steps=config.WARMUP_STEPS,
    lr_scheduler_type="linear",
    num_train_epochs=config.NUM_EPOCHS,
    eval_steps=config.EVAL_STEPS,
    save_steps=config.SAVE_STEPS,
    logging_steps=config.LOGGING_STEPS,
    eval_strategy="steps",
    predict_with_generate=True,
    generation_max_length=225,
    fp16=config.FP16,
    dataloader_num_workers=2,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to=["tensorboard"],
    push_to_hub=False,
    remove_unused_columns=False,
    label_names=["labels"],
    seed=config.SEED,
)
print("✅ Training arguments set")


✅ Training arguments set


In [11]:
trainer = WERLoggingTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
    wer_samples=wer_samples,
    wer_tokenizer=tokenizer,
)
print("✅ Trainer initialized")


2026-05-17 17:19:34.690536: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779038375.087609      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779038375.194257      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779038376.207580      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779038376.207609      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779038376.207612      57 computation_placer.cc:177] computation placer alr

✅ Trainer initialized


In [12]:
print("=" * 60)
print("🚀 STARTING FINE-TUNING (Cycle 2)")
print(f"   Train: {len(dataset['train'])} | Val: {len(dataset['test'])}")
print(f"   Effective batch: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION}")
print(f"   Epochs: {config.NUM_EPOCHS} | LR: {config.LEARNING_RATE}")
print(f"   Live WER logged every {config.LOGGING_STEPS} steps")
print("=" * 60)

torch.cuda.empty_cache()
gc.collect()
trainer.train()


🚀 STARTING FINE-TUNING
   Model:          /kaggle/input/notebooks/panditaadarsh/finetuning-the-codeswitched-whisper/whisper-small-nepali-english-cs/final-model
   Train samples:  9495
   Val samples:    500
   Batch size:     2 x 8 = 16
   Epochs:         5
   Learning rate:  1e-05
   FP16:           True
   Language force:  DISABLED (code-switching mode)


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss,Validation Loss,Wer
500,3.689879,0.272218,54.687287
1000,2.133566,0.269047,54.769042


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=225) and `max_length`(=225) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'trans

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


TrainOutput(global_step=1485, training_loss=3.028835836082998, metrics={'train_runtime': 18674.1532, 'train_samples_per_second': 2.542, 'train_steps_per_second': 0.08, 'total_flos': 1.3700591880192e+19, 'train_loss': 3.028835836082998, 'epoch': 5.0})

In [13]:
FINAL_MODEL_DIR = os.path.join(config.OUTPUT_DIR, "final-model")
print(f"💾 Saving final model to {FINAL_MODEL_DIR}...")
model.save_pretrained(FINAL_MODEL_DIR)
processor.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print("✅ Model saved!")


💾 Saving final model to /kaggle/working/whisper-small-nepali-english-cs/final-model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved!


In [ ]:
# ── Helper utilities (no external research_eval needed) ─────────────────────
import unicodedata
from collections import Counter

def edit_dist(hyp, ref):
    """Word-level Levenshtein distance."""
    n, m = len(ref), len(hyp)
    dp = list(range(n + 1))
    for j in range(1, m + 1):
        prev, dp[0] = dp[0], j
        for i in range(1, n + 1):
            tmp = dp[i]
            dp[i] = prev if hyp[j-1] == ref[i-1] else 1 + min(prev, dp[i], dp[i-1])
            prev = tmp
    return dp[n]

def word_errors_detail(hyp, ref):
    """Return sub / del / ins / cor counts via DP + back-trace."""
    n, m = len(ref), len(hyp)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): dp[i][0] = i
    for j in range(m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            dp[i][j] = dp[i-1][j-1] if ref[i-1]==hyp[j-1] else 1+min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])
    i, j = n, m
    sub = del_ = ins = cor = 0
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref[i-1] == hyp[j-1]:
            cor += 1; i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1]+1:
            sub += 1; i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i-1][j]+1:
            del_ += 1; i -= 1
        else:
            ins += 1; j -= 1
    return {"sub": sub, "del": del_, "ins": ins, "cor": cor}

def _is_script(c, name):
    try: return name in unicodedata.name(c)
    except ValueError: return False

def classify_token(t):
    d = sum(_is_script(c, "DEVANAGARI") for c in t)
    l = sum(_is_script(c, "LATIN") for c in t)
    if d > 0 and l > 0: return "mixed"
    if d > 0: return "ne"
    if l > 0: return "en"
    return "other"

def compute_cmi(text):
    tokens = [t for t in text.split() if t.isalnum()]
    counts = Counter()
    langs = []
    for t in tokens:
        lang = classify_token(t)
        if lang in ("ne", "en"):
            counts[lang] += 1; langs.append(lang)
    N = sum(counts.values())
    if N == 0: return 0.0
    switches = sum(1 for i in range(1, len(langs)) if langs[i] != langs[i-1])
    return (N - max(counts.values()) + switches) / (2 * N)

print("✅ Helper functions ready")


In [ ]:
from transformers import pipeline

# ── Validation split (mirrors the HF split) ──────────────────────────────
val_size = int(len(df) * config.TEST_SIZE)
val_df = df.sample(frac=1, random_state=config.SEED).iloc[:val_size].reset_index(drop=True)

def get_preds(model_path, label="model"):
    print(f"🔄 [{label}] Generating predictions on {len(val_df)} samples...")
    pipe = pipeline(
        "automatic-speech-recognition",
        model=model_path,
        device=0 if torch.cuda.is_available() else -1,
        chunk_length_s=30,
    )
    preds, refs = [], []
    for _, row in val_df.iterrows():
        result = pipe(row["audio_path"])
        preds.append(result["text"].strip())
        refs.append(row["transcription"].strip())
    del pipe; gc.collect(); torch.cuda.empty_cache()
    return preds, refs

base_preds,  _       = get_preds(config.BASE_MODEL,  "Base")
cycle1_preds, _      = get_preds(config.MODEL_NAME,  "Cycle1")
cycle2_preds, all_refs = get_preds(FINAL_MODEL_DIR,  "Cycle2")

print(f"✅ Predictions generated for {len(all_refs)} validation samples")


In [ ]:
def calc_all_metrics(preds, refs, label="Model"):
    t_edit = t_ref = tc_edit = tc_ref = 0
    ts = td = ti = tc = 0
    ne_ref = en_ref = 0
    cmis_ref, cmis_pred = [], []

    for p, r in zip(preds, refs):
        rw, pw = r.split(), p.split()
        t_ref += len(rw);  t_edit += edit_dist(pw, rw)
        rc, pc = list(r.replace(" ","")), list(p.replace(" ",""))
        tc_ref += len(rc); tc_edit += edit_dist(pc, rc)
        for t in rw:
            l = classify_token(t)
            if l == "ne": ne_ref += 1
            elif l == "en": en_ref += 1
        d = word_errors_detail(pw, rw)
        ts += d["sub"]; td += d["del"]; ti += d["ins"]; tc += d["cor"]
        cmis_ref.append(compute_cmi(r)); cmis_pred.append(compute_cmi(p))

    return {
        "label": label,
        "wer": t_edit / max(t_ref,1) * 100,
        "cer": tc_edit / max(tc_ref,1) * 100,
        "ne_tokens_ref": ne_ref, "en_tokens_ref": en_ref,
        "sub": ts, "del": td, "ins": ti, "cor": tc,
        "mean_cmi_ref":  sum(cmis_ref)/max(len(cmis_ref),1),
        "mean_cmi_pred": sum(cmis_pred)/max(len(cmis_pred),1),
    }

m_base   = calc_all_metrics(base_preds,   all_refs, "Whisper-Small (base)")
m_cycle1 = calc_all_metrics(cycle1_preds, all_refs, "Cycle 1")
m_cycle2 = calc_all_metrics(cycle2_preds, all_refs, "Cycle 2")

print("✅ Metrics computed for all three models")


In [ ]:
print("\n" + "="*75)
print("  TABLE 1: ASR Performance Comparison (Code-Switched Nepali-English)")
print("="*75)
print(f"{'Model':<30} {'WER%':>8} {'CER%':>8}")
print("-" * 75)
for m in [m_base, m_cycle1, m_cycle2]:
    print(f"{m['label']:<30} {m['wer']:>8.2f} {m['cer']:>8.2f}")
print("="*75)

print(f"\n{'─'*75}")
print("  TABLE 2: Error Analysis (Cycle 2)")
print(f"{'─'*75}")
total_err = m_cycle2["sub"] + m_cycle2["del"] + m_cycle2["ins"]
print(f"{'Error Type':<25} {'Count':>10} {'% of Errors':>15}")
for name, key in [("Substitutions","sub"),("Deletions","del"),("Insertions","ins")]:
    print(f"{name:<25} {m_cycle2[key]:>10,} {m_cycle2[key]/max(total_err,1)*100:>14.1f}%")

print(f"\n{'─'*75}")
print("  TABLE 3: Code-Mixing Statistics")
print(f"{'─'*75}")
print(f"{'Mean CMI (ref)':<40} {m_cycle2['mean_cmi_ref']:>15.4f}")
print(f"{'Mean CMI (pred)':<40} {m_cycle2['mean_cmi_pred']:>15.4f}")


In [ ]:
import random
random.seed(42)

print("\n" + "="*75)
print("  TABLE 4: Qualitative Examples — Cycle 2 Model")
print("="*75)

idxs = random.sample(range(len(cycle2_preds)), min(4, len(cycle2_preds)))
for k, i in enumerate(idxs):
    ref = all_refs[i]; hyp = cycle2_preds[i]
    print(f"\n── Example {k+1} ──")
    print(f"REF: {ref[:120]}")
    print(f"HYP: {hyp[:120]}")
    ref_langs = [classify_token(t) for t in ref.split()[:15]]
    print(f"LNG: {' '.join(ref_langs)}")
    print(f"EDIT DIST (word): {edit_dist(hyp.split(), ref.split())}")
print("\n" + "="*75)


In [ ]:
print("\n" + "="*60)
print("🧪 QUICK INFERENCE TEST")
print("="*60)

pipe = pipeline(
    "automatic-speech-recognition",
    model=FINAL_MODEL_DIR,
    device="cuda:0" if torch.cuda.is_available() else "cpu",
    chunk_length_s=30,
)

test_paths = df["audio_path"].sample(3, random_state=42).tolist()
for ap in test_paths:
    result = pipe(ap)
    fn = os.path.basename(ap)
    gt = df[df["audio_path"] == ap]["transcription"].values[0]
    print(f"\n🎵 {fn}")
    print(f"   📝 GT:   {gt[:120]}")
    print(f"   🤖 Pred: {result['text'][:120]}")

del pipe; gc.collect(); torch.cuda.empty_cache()
print("\n🎉 FINE-TUNING COMPLETE!")
